# 04 — Structured Output (Extraction)

Getting the model to return **typed, validated data** (not free text you then have to regex-parse) is one of the most practically useful LangChain patterns — great for extracting fields from documents, forms, emails, etc.

**Deprecated / avoid:** manually prompting "respond only in JSON" and parsing with `json.loads` — brittle, breaks silently.

**Current:** `.with_structured_output(YourPydanticModel)` on a chat model. The model provider's native structured-output / tool-calling support does the validation for you.


In [ ]:
import os
from getpass import getpass
from langchain.chat_models import init_chat_model

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")
MODEL_ID = "openai:gpt-4.1-mini"
model = init_chat_model(MODEL_ID, temperature=0)

## 1. Define your schema with Pydantic

Field descriptions matter here too — they're passed to the model as instructions for what each field means.


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class ResignationEmailDetails(BaseModel):
    """Structured details extracted from a resignation-related email."""
    employee_name: Optional[str] = Field(None, description="Name of the employee, if mentioned")
    last_working_day: Optional[str] = Field(None, description="Last working day, in any format mentioned")
    notice_period_days: Optional[int] = Field(None, description="Notice period length in days, if stated")
    tone: str = Field(description="Overall tone: formal, friendly, or terse")
    requires_action: bool = Field(description="Whether the recipient needs to take an action")

structured_model = model.with_structured_output(ResignationEmailDetails)

## 2. Run extraction

Notice the output is a real Pydantic object — you get autocomplete, type checking, and validation, not a dict you have to trust blindly.


In [ ]:
email_text = """
Hi Team,

I am writing to formally resign from my position, effective as my last working day
will be October 20, 2026. As per my contract, this reflects the standard 90-day notice
period. Please let HR know so the F&F process can be initiated in time.

Best,
Ankur
"""

result = structured_model.invoke(f"Extract the details from this email:\n\n{email_text}")
print(result)
print()
print("Type:", type(result))
print("Last working day:", result.last_working_day)
print("Requires action:", result.requires_action)

## 3. Structured output inside an LCEL chain

You can drop `.with_structured_output(...)` right into a pipe chain, same as any other step.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract structured details from the email below."),
    ("human", "{email}"),
])

chain = extraction_prompt | structured_model
result2 = chain.invoke({"email": email_text})
print(result2)

## 4. Structured output as an agent's final response format

`create_agent` accepts a `response_format` argument — the agent can use tools freely to gather info, but its *final* answer is forced into your schema. Useful when an agent needs to research something but you still want a clean, typed result at the end.


In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool

class StockSummary(BaseModel):
    """A structured summary of a stock lookup."""
    ticker: str = Field(description="The stock ticker symbol")
    price: float = Field(description="Current price")
    recommendation: str = Field(description="One of: buy, hold, sell")

@tool
def get_price(ticker: str) -> float:
    """Get the current price for a ticker symbol."""
    return {"TCS": 4123.50, "INFY": 1567.20}.get(ticker.upper(), 0.0)

structured_agent = create_agent(
    model=model,
    tools=[get_price],
    response_format=StockSummary,
)

out = structured_agent.invoke({"messages": [{"role": "user", "content": "Look up TCS and give me a recommendation."}]})
print(out["structured_response"])
print(type(out["structured_response"]))

---
### Key takeaways
- `.with_structured_output(PydanticModel)` gives you validated, typed data — no manual JSON parsing.
- Field `description`s are instructions to the model — write them clearly.
- Works standalone, inside an LCEL chain, or as an agent's `response_format`.
- This is the reliable way to do extraction, form-filling, or any "turn free text into structured data" task.

**Next:** `05_rag_pipeline.ipynb` — retrieval-augmented generation over your own documents.
